# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Saad-Imran-Toori/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**My lane: Lane 4 — CTR / Engagement Opportunity Scoring.**

When I ran the starter notebooks, one observation stuck with me: search demand alone tells you almost nothing about traffic (correlation between `search_volume` and `impressions_90d` is ~0.001 in this slice). So where does traffic actually get won or lost? One measurable place is the gap between being *shown* and being *clicked*. Two pages at the same ranking position can have very different CTRs — and the page below its tier's typical CTR is, directionally, leaving clicks on the table. I picked this lane because the action it drives (reviewing a title/snippet) is cheap, the opportunity is measurable per page, and it is distinct from the refresh pipeline the reference scripts already implement. Provisional until Week 4, as allowed.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Locate repo root and load the starter data once for the whole notebook.
import os
import pandas as pd
import numpy as np

while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The observation that pointed me at this lane (from notebook 01):
corr = df["search_volume"].corr(df["impressions_90d"])
print(f"Pages: {len(df):,} | corr(search_volume, impressions_90d) = {corr:.3f}")
print("Demand alone doesn't explain traffic -> look at where shown pages fail to convert to clicks.")


Pages: 30,000 | corr(search_volume, impressions_90d) = 0.001
Demand alone doesn't explain traffic -> look at where shown pages fail to convert to clicks.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**The question:** which visible pages under-capture clicks relative to pages at the same position tier, and deserve a metadata/content review first?

**The frame (filled honestly):** For a FlyRank content editor, deciding *which visible pages get a title/snippet/intent review this week*, I will build a **ranked review queue with reason codes** from the starter dataset (and later the warehouse), scoring each page by how far its CTR sits **below the typical CTR of its own position tier** (with volume floors so one click can't fake a signal). Success is measured by **precision@K** — of the top K pages my queue flags, how many a sensible reviewer would agree deserve the review.

**Unit of analysis:** one content page (one row = one pseudonymized page with trailing-90-day metrics).

**Output:** a ranked queue + a per-page reason code (e.g. "p4–10 tier, 12k impressions, CTR 0.3% vs tier median 0.23% — below expectation at volume").

**Who acts, and how:** an editor takes the top of the queue and rewrites the title/meta description, improves the snippet structure, or flags an intent mismatch. Review capacity is the scarce resource — maybe tens of pages a week, not thousands — which is exactly why a *ranking* matters.

**Cost of a wrong call:** a false positive wastes a slot of scarce editor time on a page that was fine (and a pointless rewrite can even hurt a working title). A false negative leaves cheap clicks unclaimed. Since the action is cheap and reversible, false negatives at the top of the queue matter more than occasional false positives — but a queue full of noise destroys editor trust, so precision at the top is the metric.

**Why data/ML at all, and why not just one rule?** "Low CTR = bad title" fails immediately: expected CTR depends on position, volume, intent, and content type all at once. A fair expectation has to be *conditional* — that's a statistical model by definition, even a simple one. I'll still build the transparent rule first as the baseline and let anything fancier earn its keep against it.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Scale check: why a ranked queue (not a to-do list) is the right output shape.
visible = df[(df["avg_position"] > 0) & (df["impressions_90d"] >= 100)]
print(f"Visible pages (position data + >=100 impressions): {len(visible):,}")
print(f"A team reviewing ~50 pages/week would need {len(visible)/50:.0f} weeks to see them all.")
print("-> the decision is WHICH pages first; ranking is the product.")


Visible pages (position data + >=100 impressions): 22,006
A team reviewing ~50 pages/week would need 440 weeks to see them all.
-> the decision is WHICH pages first; ranking is the product.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Three numbers, computed below with the dataset's own gotchas respected (`ctr` is a ×100 percentage; `avg_position = 0` means "no position data" and is excluded; tier medians are only read above a volume floor):

1. **How many pages are even in scope** — visible pages with position data and real impressions.
2. **The tier expectation exists and differs by tier** — median CTR by `position_tier` at a ≥500-impressions floor, which is the "fair comparison" the whole lane stands on.
3. **The opportunity is big** — how many in-scope pages sit below their own tier's median CTR, and a directional estimate of the 90-day clicks that gap represents.


In [3]:
# This cell is for CODE (numbers, a query, a check).
FLOOR = 500  # impressions_90d floor: below this, one click moves CTR too much to trust

scope = df[(df["avg_position"] > 0) & (df["impressions_90d"] >= FLOOR)].copy()
print(f"1) In-scope pages (position data + >= {FLOOR} impressions/90d): {len(scope):,} of {len(df):,}")

tier_median = scope.groupby("position_tier")["ctr"].median()
print("\n2) Median CTR (%) by position tier at this floor:")
print(tier_median.round(2).to_string())

scope["tier_median_ctr"] = scope["position_tier"].map(tier_median)
scope["ctr_gap_pp"] = scope["tier_median_ctr"] - scope["ctr"]          # positive = below expectation
below = scope[scope["ctr_gap_pp"] > 0]
lost_clicks = (below["ctr_gap_pp"] / 100 * below["impressions_90d"]).sum()
print(f"\n3) Pages below their own tier's median CTR: {len(below):,} "
      f"({len(below)/len(scope):.0%} of in-scope)")
print(f"   Directional size of the gap: ~{lost_clicks:,.0f} clicks/90d if every below-median page")
print(f"   merely reached its tier median (an upper-bound thought experiment, not a promise).")


1) In-scope pages (position data + >= 500 impressions/90d): 16,726 of 30,000

2) Median CTR (%) by position tier at this floor:
position_tier
deep        0.00
page_1      0.24
page_3_5    0.09
striking    0.17
top_3       0.20

3) Pages below their own tier's median CTR: 7,950 (48% of in-scope)
   Directional size of the gap: ~61,164 clicks/90d if every below-median page
   merely reached its tier median (an upper-bound thought experiment, not a promise).


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**I can say:** "we *observed* that page X's CTR sits N points below the *median* CTR of pages in the same position tier at a comparable volume, so it is a *reasonable candidate* for a title/snippet review." All results are **observed, directional, decision-support** — a prioritization aid for a human reviewer.

**I cannot and will not say:** that a low CTR *proves* the title is bad (intent mismatch, brand queries, SERP features, or seasonality can all depress CTR); that fixing the title *will* recover the estimated clicks (the gap estimate is an upper-bound thought experiment on one 90-day snapshot); anything *causal* — this is observational data, nobody ran an experiment; or anything about "predicting Google's algorithm."

**Known limits I'll carry forward:** one snapshot (no time dimension yet — the warehouse adds it); tier medians are sensitive to the volume floor (stated explicitly every time, per the data dictionary's warning); `ctr` is a ×100 percentage and `avg_position = 0` rows are excluded as "no data", both handled above; and comparisons are only ever *within* a position tier, never across tiers.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Why the volume floor is not optional: at low volume, one click moves CTR by whole points.
low = df[(df["avg_position"] > 0) & (df["impressions_90d"] > 0) & (df["impressions_90d"] < 100)]
one_click_pp = (1 / low["impressions_90d"] * 100).median()
print(f"Pages with 1-99 impressions: {len(low):,}")
print(f"For the median such page, ONE extra click moves CTR by {one_click_pp:.1f} percentage points.")
print("-> any CTR-gap claim below a volume floor would be reading noise; the floor stays.")


Pages with 1-99 impressions: 6,789
For the median such page, ONE extra click moves CTR by 5.9 percentage points.
-> any CTR-gap claim below a volume floor would be reading noise; the floor stays.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.